# Geração de Textos Sintéticos (Qwen 2.5) a partir das Sementes `seed_discarded`

Implementa o fluxo de extração factual e geração de artigos sintéticos espelhados, persistindo os registros em formato estruturado.

**Pipeline**
1. Lê o `preprocessed_corpus.jsonl` (todos os `provenance_type`) e **filtra somente** `provenance_type == "seed_discarded"` (esperado: 1.500 registros, 75 por estrato `(conferência, ano)` no pré-2020).
2. Preserva as sementes em `data/processed/seeds_discarded.jsonl` (nunca sobrescreve nem apaga).
3. **Etapa 1** - extração de fatos técnicos (JSON) com `Qwen/Qwen2.5-7B-Instruct` via vLLM.
4. **Etapa 2** - redação sintética de Abstract / Introdução / Conclusão a partir **apenas dos fatos**.
5. Valida (tamanho mínimo, inglês ≥ 0.98, sem cópia literal da semente), com *retries* e *checkpoint* retomável.
6. Salva `data/processed/synthetic_ai.jsonl` (`provenance_type = "synthetic_ai"`, `label = 1`).

> Edite apenas a célula **Configuração**.

## 1. Configuração

In [ ]:
# ============================== CONFIGURAÇÃO ==============================
CORPUS_PATH = "/kaggle/input/datasets/gustavofcunha/artigos/collected_corpus.jsonl"   # <<< INSIRA AQUI o caminho do seu arquivo

# Saídas estruturadas do fluxo de geração
OUTPUT_PATH        = "/kaggle/working/synthetic_ai.jsonl"
FACTS_OUTPUT_PATH  = "/kaggle/working/extracted_facts.jsonl"
SEEDS_ARCHIVE_PATH = "/kaggle/working/seeds_discarded.jsonl"
CACHE_DIR          = "/kaggle/working/_synthetic_cache_v2"   # checkpoints (retomada automática)

# Filtro das sementes
SEED_PROVENANCE   = "seed_discarded"
EXPECTED_SEEDS    = 1500     # 20 estratos x 75
STRICT_COUNTS     = True     # True: aborta se o TOTAL de sementes divergir de EXPECTED_SEEDS
STRICT_STRATA     = False    # True: aborta se algum estrato (conferência x ano) != 75. Padrão False: a distribuição
                             # real pode ser desigual (ex.: ICLR 2015 tem poucos artigos); só é exibida como aviso.
EXPECTED_PER_STRATUM = 75    # usado apenas para o aviso/tabela de desvios

# Backend: "vllm" (real) ou "mock" (teste de fumaça sem GPU)
BACKEND = "vllm"

# Modelo oficial (seção 2.4). Em GPU de 16 GB (T4) os pesos fp16 de 7B (~15 GB) NÃO cabem
# com KV-cache; por isso, se a VRAM for < 20 GB, usamos a variante AWQ 4-bit do MESMO modelo.
MODEL_NAME            = "Qwen/Qwen2.5-7B-Instruct"
AWQ_MODEL_NAME        = "Qwen/Qwen2.5-7B-Instruct-AWQ"
AUTO_AWQ_ON_SMALL_GPU = True
MAX_MODEL_LEN         = 16384   # prompt + saída (Qwen2.5 suporta 32k; KV-cache GQA é leve)
GPU_MEM_UTIL          = 0.90
MAX_NUM_SEQS          = 32      # concorrência no vLLM (evita thrashing de KV-cache em T4)
ENFORCE_EAGER         = False   # True: desliga CUDA graphs/compilação (mais lento, mas menos memória e menos falhas de init)
MAX_RUNTIME_HOURS     = 10.0    # para a geração com folga antes do limite da sessão (Kaggle/Colab); retoma do cache

# Amostragem
SEED_RNG            = 42
EXTRACT_TEMPERATURE = 0.2    # etapa 1: factual
DRAFT_TEMPERATURE   = 0.7    # etapa 2: redação
DRAFT_TOP_P         = 0.9
# max_tokens com MUITA folga (saída esperada: etapa 1 ~1-2k tokens; etapa 2 ~2-3k tokens).
# Se algo ainda for cortado (finish_reason == "length"), o registro é rejeitado e refeito  -  nunca aceito truncado.
EXTRACT_MAX_TOKENS  = 4096
DRAFT_MAX_TOKENS    = 8192
PROMPT_SAFETY_MARGIN = 512   # tokens reservados para o template de chat
BATCH_SIZE          = 128    # prompts por chamada ao vLLM (checkpoint a cada lote)
MAX_ROUNDS          = 4      # tentativas por registro

# Elegibilidade (mesmos limiares da seção 2.1)
MIN_ABSTRACT_WORDS = 50
MIN_INTRO_WORDS    = 200
MIN_CONC_WORDS     = 50
MIN_ENGLISH_PROB   = 0.98
MAX_8GRAM_OVERLAP  = 0.15    # fração máx. de 8-gramas copiados da semente

PROMPT_ID_EXTRACT = "prompt1_fact_extraction_v2"
PROMPT_ID_DRAFT   = "prompt2_scientific_mirror_drafting_v2"

# provenance_type gravado nos registros sintéticos
OUTPUT_PROVENANCE_TYPE = "synthetic_ai"

INSTALL_DEPS = True          # instala pacotes ausentes (Colab/venv com internet)
# ==========================================================================

## 2. Dependências

In [ ]:
import importlib, importlib.util, subprocess, sys

REQUIRED = {"pandas": "pandas", "tqdm": "tqdm", "langdetect": "langdetect"}
if BACKEND == "vllm":
    REQUIRED["vllm"] = "vllm"          # traz torch/transformers compatíveis

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing and INSTALL_DEPS:
    print("Instalando:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    importlib.invalidate_caches()
    if "vllm" in missing:
        print("vLLM recém-instalado: se aparecer erro de import/CUDA, reinicie o runtime e rode de novo.")
still = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
assert not still, f"Dependências ausentes: {still}"
print("Dependências OK")

In [ ]:
import hashlib, json, os, random, re, time
from collections import Counter, defaultdict
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm
from langdetect import DetectorFactory, detect_langs

DetectorFactory.seed = 42
random.seed(SEED_RNG)
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

def wc(text):
    return len(re.findall(r"\S+", text or ""))

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON inválido em {path}, linha {i}: {e}") from e

def append_jsonl(path, rows):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

## 3. Carga e filtro das sementes (`seed_discarded`)

In [ ]:
assert os.path.isfile(CORPUS_PATH), f"Arquivo não encontrado: {CORPUS_PATH} (ajuste CORPUS_PATH)"

prov_counter, seeds = Counter(), []
for rec in read_jsonl(CORPUS_PATH):
    p = rec.get("provenance_type")
    prov_counter[p] += 1
    if p == SEED_PROVENANCE:
        seeds.append(rec)

print("provenance_type no arquivo:", dict(prov_counter))
print(f"Sementes selecionadas ({SEED_PROVENANCE}): {len(seeds)}")

# --- validações ---
ids = [s["paper_id"] for s in seeds]
assert len(ids) == len(set(ids)), "paper_id duplicado entre as sementes"
required_fields = ["paper_id", "title", "abstract", "introduction", "conclusion"]
bad = [s.get("paper_id") for s in seeds if any(not s.get(k) for k in required_fields)]
assert not bad, f"{len(bad)} sementes com campos obrigatórios vazios, ex.: {bad[:3]}"
# conferência/ano são usados SÓ para validar os estratos aqui; nunca entram nos prompts nem nos registros sintéticos
assert all(s["publication_year"] <= 2019 for s in seeds), "Há semente com ano >= 2020 (viola o marco temporal)"

strata = Counter((s["conference"], s["publication_year"]) for s in seeds)
display(pd.Series(strata).unstack(fill_value=0)) if "display" in globals() else print(pd.Series(strata).unstack(fill_value=0))

if len(seeds) != EXPECTED_SEEDS:
    m = f"esperado {EXPECTED_SEEDS} sementes, encontrado {len(seeds)}"
    if STRICT_COUNTS:
        raise AssertionError(m)
    print("AVISO:", m)
else:
    print(f"Total de sementes confere: {len(seeds)}.")

uneven = {k: v for k, v in sorted(strata.items()) if v != EXPECTED_PER_STRATUM}
if uneven:
    m = (f"{len(uneven)} de {len(strata)} estratos não têm exatamente {EXPECTED_PER_STRATUM} sementes "
         f"(min={min(strata.values())}, max={max(strata.values())}). Isso não afeta a geração: "
         f"cada semente gera uma contraparte, preservando a mesma distribuição.")
    if STRICT_STRATA:
        raise AssertionError(m)
    print("AVISO:", m)

# --- preservação para auditoria (nunca sobrescrever/apagar) ---
if os.path.exists(SEEDS_ARCHIVE_PATH):
    print(f"{SEEDS_ARCHIVE_PATH} já existe  -  mantido intacto.")
else:
    append_jsonl(SEEDS_ARCHIVE_PATH, seeds)
    print(f"Sementes preservadas em {SEEDS_ARCHIVE_PATH}")

## 4. Prompts oficiais (seção 3.4)  -  recebem SOMENTE título, abstract, introdução e conclusão

In [ ]:
EXTRACT_SYSTEM = (
    "You are an expert scientific analyst specializing in machine learning and computational linguistics. "
    "Your task is to analyze the provided raw paper sections and extract their core technical facts into a concise, "
    "factual JSON structure. Silently normalize PDF extraction artifacts (such as broken line-wrap hyphens or ligatures) "
    "and ignore non-prose noise (figure/table captions, conference headers or years, author emails, footnote markers, "
    "pseudocode listings, and any leaked reference lists or downstream sections). Do not include introductory filler, "
    "conversational pleasantries, or stylistic evaluations."
)

EXTRACT_USER_TEMPLATE = """Extract the core technical substance from the raw scientific paper sections below into a structured JSON object.

[INPUT SECTIONS]
Title: {title}
Abstract: {abstract}
Introduction: {introduction}
Conclusion: {conclusion}

[OUTPUT FORMAT]
Output strictly a valid JSON object matching this schema (exclude paper-level metadata such as venue names, publication years of this paper, author affiliations, or figure/table numbers):
{
  "problem_statement": "The core research question and technical challenge addressed.",
  "core_methodology": "The key proposed model, theoretical framework, and formal mechanisms (include equations and variable notations in LaTeX $...$ only if mathematical expressions or formal symbols appear in the source text).",
  "empirical_findings": "The quantitative benchmarks, experimental evidence, datasets, and verified results.",
  "related_work_and_baselines": "All cited prior works, foundational claims, and baseline methods from the text, pairing each exact citation marker as written (whether author-year such as Author et al. (Year) / (Author and Author, Year), or numeric brackets such as [1], [2, 5]) with a concise paraphrased summary (in your own words, not verbatim prose) of the specific claim, method, or finding that citation supports.",
  "resources_and_urls": "Any public code repository links, dataset URLs, or project artifacts explicitly mentioned in the prose (excluding author emails or publisher footers).",
  "main_conclusions": "The concluding takeaways, practical implications, and future research avenues."
}"""

DRAFT_SYSTEM = (
    "You are an expert AI researcher drafting a peer-reviewed academic conference paper "
    "in machine learning or computational linguistics. Using strictly the provided technical facts, write three complete, "
    "formal, and authoritative scientific paper sections: Abstract, Introduction, and Conclusion.\n"
    "Maintain natural academic prose, clear paragraph transitions, and varied sentence structure typical of "
    "published peer-reviewed papers. Strictly respect the requested word-count target for each section without padding "
    "or truncating prematurely. Never include venue names, author details, figure/table numbers (e.g., Figure 1, Table 2), "
    "or PDF formatting artifacts. Output strictly a valid JSON object containing the exact keys \"abstract\", \"introduction\", and \"conclusion\"."
)

DRAFT_USER_TEMPLATE = """Using exclusively the technical facts provided below, write a new, complete, and formal academic paper comprising an Abstract, Introduction, and Conclusion.

[TECHNICAL FACTS]
Problem: {problem_statement}
Methodology & Math: {core_methodology}
Prior Literature & Baselines: {related_work_and_baselines}
Findings: {empirical_findings}
Resources & URLs: {resources_and_urls}
Conclusions: {main_conclusions}

[SECTION REQUIREMENTS]
1. abstract: A dense, self-contained summary strictly within 130 to 170 words (target ~145 words) framing the problem, the proposed approach, key technical mechanisms, and principal empirical findings.
2. introduction: A comprehensive introduction strictly within 550 to 750 words (target ~600 words, developed across 4 to 6 substantial paragraphs) contextualizing the domain, motivating and formalizing the problem, and actively grounding background claims in Prior Literature & Baselines. Replicate the exact citation markers provided in Prior Literature & Baselines alongside the paraphrased claims they support: preserve nominal author-year style when provided (mixing narrative, e.g., Author et al. (Year), and parenthetical, e.g., (Author et al., Year)), and replicate the exact numeric bracket citations (e.g., [1], [2, 5]) when numeric citations are provided. Never invent author names, years, or citation numbers not present in the facts. Explain the proposed methodology clearly (using inline LaTeX $...$ only when mathematical formulations are present in the facts), mention resource or repository links if provided, and conclude the introduction in cohesive academic prose synthesizing the paper's main findings and contributions without forcing a templated bulleted list.
3. conclusion: A formal closing section strictly within 180 to 250 words (target ~210 words, 1 to 2 substantial paragraphs) synthesizing the main findings, noting limitations or scope boundaries, and outlining future directions.

[OUTPUT FORMAT]
Output strictly a valid JSON object:
{
  "abstract": "...",
  "introduction": "...",
  "conclusion": "..."
}"""

FACT_KEYS = ["problem_statement", "core_methodology", "empirical_findings",
             "related_work_and_baselines", "resources_and_urls", "main_conclusions"]
REQUIRED_FACT_KEYS = ["problem_statement", "core_methodology", "main_conclusions"]

def fill(template, **values):
    """Substitui {campo} por str.replace (o template contém chaves JSON literais, então não usamos .format)."""
    out = template
    for k, v in values.items():
        out = out.replace("{" + k + "}", v)
    return out

def build_extract_messages(seed, intro_text=None):
    # SOMENTE title/abstract/introduction/conclusion  -  nenhum outro metadado da semente
    user = fill(EXTRACT_USER_TEMPLATE, title=seed["title"], abstract=seed["abstract"],
                introduction=seed["introduction"] if intro_text is None else intro_text,
                conclusion=seed["conclusion"])
    return [{"role": "system", "content": EXTRACT_SYSTEM}, {"role": "user", "content": user}]

def build_draft_messages(facts):
    # SOMENTE os fatos extraídos (sem título, conferência, ano, autores, URL, nem tamanhos da semente)
    user = fill(DRAFT_USER_TEMPLATE, **{k: facts[k] for k in FACT_KEYS})
    return [{"role": "system", "content": DRAFT_SYSTEM}, {"role": "user", "content": user}]

## 5. Backend de geração (vLLM / mock)

In [ ]:
class MockBackend:
    """Backend de fumaça: não usa GPU. Só para validar o pipeline."""
    name = "mock"
    model_name = "mock-llm"
    def count_tokens(self, messages):
        return int(sum(wc(m["content"]) for m in messages) * 1.5)
    def chat(self, batch_messages, temperature, top_p, max_tokens, seed=None):
        """Retorna lista de (texto, truncado: bool)."""
        out = []
        for msgs in batch_messages:
            u = msgs[-1]["content"]
            if "Extract the core technical substance" in u:
                f = {k: "A technical statement about the paper, including a loss $\\mathcal{L} = \\beta x$." for k in FACT_KEYS}
                out.append((json.dumps(f), False))
            else:
                s = ("We study a learning problem and propose a method that improves performance on standard benchmarks. "
                     "Our approach combines a simple model with careful optimization and a thorough empirical evaluation. ")
                d = json.dumps({"abstract": s * 5, "introduction": s * 20 + " We minimize $\\beta \\frac{1}{2}$ and the loss.\n- We propose X.\n- We show Y.", "conclusion": s * 5})
                d = d.replace("\\\\beta", "\\beta").replace("\\\\frac", "\\frac")   # simula LaTeX SEM escape correto
                out.append((d, False))
        return out

class VLLMBackend:
    name = "vllm"
    def __init__(self):
        import torch
        from vllm import LLM
        assert torch.cuda.is_available(), "GPU CUDA não disponível (Colab: Ambiente de execução > GPU T4; Kaggle: Accelerator > GPU T4 x2)."
        cap = torch.cuda.get_device_capability(0)
        assert cap >= (7, 0), (f"GPU {torch.cuda.get_device_name(0)} (compute {cap[0]}.{cap[1]}) não é suportada pelo vLLM "
                               f"(mínimo 7.0). No Kaggle, troque P100 por T4; no Colab use T4/L4/A100.")
        import vllm
        print(f"vllm {vllm.__version__} | torch {torch.__version__} | CUDA {torch.version.cuda}")
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        use_awq = AUTO_AWQ_ON_SMALL_GPU and vram < 20
        self.model_name = AWQ_MODEL_NAME if use_awq else MODEL_NAME
        major = torch.cuda.get_device_capability(0)[0]
        dtype = "float16" if (use_awq or major < 8) else "bfloat16"   # T4 não suporta bf16
        print(f"GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB) -> {self.model_name} dtype={dtype}")
        kw = dict(model=self.model_name, dtype=dtype, max_model_len=MAX_MODEL_LEN, max_num_seqs=MAX_NUM_SEQS,
                  gpu_memory_utilization=GPU_MEM_UTIL, seed=SEED_RNG, enforce_eager=ENFORCE_EAGER)
        if use_awq:
            kw["quantization"] = "awq"
        self.llm = LLM(**kw)
        self.tok = self.llm.get_tokenizer()

    def count_tokens(self, messages):
        return len(self.tok.encode("\n".join(m["content"] for m in messages)))

    def chat(self, batch_messages, temperature, top_p, max_tokens, seed=None):
        from vllm import SamplingParams
        sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_tokens)  # semente global em LLM(seed=...)
        outs = self.llm.chat(batch_messages, sp, use_tqdm=False)
        return [(o.outputs[0].text, o.outputs[0].finish_reason == "length") for o in outs]

backend = MockBackend() if BACKEND == "mock" else VLLMBackend()
print("Backend:", backend.name, "| modelo:", backend.model_name)

## 6. Parsing e validação das saídas

In [ ]:
# Escapes JSON válidos que preservamos. Qualquer outra barra (LaTeX: \beta, \frac, \theta, \nabla, \alpha, \{ ...)
# é duplicada, para que "\beta" não vire backspace+"eta" e "\frac" não vire form-feed+"rac".
_JSON_ESC = re.compile(r'\\\\|\\["/]|\\u[0-9a-fA-F]{4}|\\[bfnrt](?![a-z])|\\')

def fix_json_escapes(t):
    return _JSON_ESC.sub(lambda m: m.group(0) if len(m.group(0)) > 1 else "\\\\", t)

def _loads(t):
    obj = json.loads(fix_json_escapes(t), strict=False)
    return obj if isinstance(obj, dict) else None

def parse_json_obj(text):
    """Extrai o primeiro objeto JSON de uma resposta do LLM (tolera cercas ```, texto extra e LaTeX sem escape)."""
    if text is None:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S)
    try:
        return _loads(t)
    except json.JSONDecodeError:
        pass
    start = t.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False
        for j in range(start, len(t)):          # casamento de chaves ciente de strings
            ch = t[j]
            if in_str:
                if esc: esc = False
                elif ch == "\\": esc = True
                elif ch == '"': in_str = False
            elif ch == '"': in_str = True
            elif ch == "{": depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return _loads(t[start:j + 1])
                    except json.JSONDecodeError:
                        break
        start = t.find("{", start + 1)
    return None

def _as_text(v):
    if v is None: return ""
    if isinstance(v, str): return v.strip()
    if isinstance(v, list): return "; ".join(_as_text(x) for x in v if _as_text(x))
    if isinstance(v, dict): return json.dumps(v, ensure_ascii=False)
    return str(v).strip()

def clean_facts(obj):
    if not obj:
        return None
    facts = {k: _as_text(obj.get(k)) for k in FACT_KEYS}
    if any(not facts[k] for k in REQUIRED_FACT_KEYS):
        return None
    for k in FACT_KEYS:
        if not facts[k]:
            facts[k] = "None mentioned."
    return facts

def clean_text(s):
    """Preserva LaTeX, citações e a lista '- We propose...' (como pedido no prompt); só normaliza espaços/markdown."""
    s = re.sub(r"```\w*", "", s)
    s = s.replace("**", "")
    return re.sub(r"\s+", " ", s).strip()

def english_prob(text):
    try:
        return next((l.prob for l in detect_langs(text) if l.lang == "en"), 0.0)
    except Exception:
        return 0.0

def ngram_set(text, n=8):
    w = re.findall(r"\w+", text.lower())
    return {" ".join(w[i:i + n]) for i in range(max(0, len(w) - n + 1))}

def overlap_with_seed(gen_text, seed_text, n=8):
    g = ngram_set(gen_text, n)
    return len(g & ngram_set(seed_text, n)) / len(g) if g else 0.0

def validate_draft(obj, seed):
    """Retorna (sections|None, motivo_falha|None)."""
    if not obj:
        return None, "json_invalido"
    secs = {}
    for k in ("abstract", "introduction", "conclusion"):
        v = obj.get(k)
        if not isinstance(v, str):
            return None, f"sem_{k}"
        secs[k] = clean_text(v)
    if wc(secs["abstract"]) < MIN_ABSTRACT_WORDS: return None, "abstract_curto"
    if wc(secs["introduction"]) < MIN_INTRO_WORDS: return None, "intro_curta"
    if wc(secs["conclusion"]) < MIN_CONC_WORDS: return None, "conclusao_curta"
    full = " ".join(secs.values())
    if re.search(r"(?i)\b(here is|here's|as an ai|as a language model)\b", full): return None, "meta_texto"
    if english_prob(full) < MIN_ENGLISH_PROB: return None, "nao_ingles"
    seed_full = " ".join([seed["abstract"], seed["introduction"], seed["conclusion"]])
    if overlap_with_seed(full, seed_full) > MAX_8GRAM_OVERLAP: return None, "copia_da_semente"
    return secs, None

# auto-teste das funções
assert parse_json_obj('```json\n{"a": 1}\n```') == {"a": 1}
assert parse_json_obj('Sure! {"a": {"b": 2}} done') == {"a": {"b": 2}}
assert parse_json_obj("sem json") is None
# LaTeX sem escape NÃO pode virar caractere de controle
_t = parse_json_obj('{"a": "$\\beta + \\frac{1}{2} \\theta \\nabla \\text{x} \\alpha$", "b": "linha1\\nThe fim"}')
assert _t["a"] == r"$\beta + \frac{1}{2} \theta \nabla \text{x} \alpha$", _t["a"]
assert _t["b"] == "linha1\nThe fim"
assert parse_json_obj(r'{"a": "$\\beta$"}')["a"] == r"$\beta$"          # já escapado corretamente
assert parse_json_obj('{"a": "quebra\nreal"}')["a"] == "quebra\nreal"      # newline literal dentro da string
print("Parsers OK")

## 7. Geração com checkpoint e retries

In [ ]:
FACTS_CACHE = os.path.join(CACHE_DIR, "facts.jsonl")
DRAFTS_CACHE = os.path.join(CACHE_DIR, "drafts.jsonl")

facts_by_id = {r["paper_id"]: r["facts"] for r in read_jsonl(FACTS_CACHE)} if os.path.exists(FACTS_CACHE) else {}
drafts_by_id = {r["paper_id"]: r for r in read_jsonl(DRAFTS_CACHE)} if os.path.exists(DRAFTS_CACHE) else {}
seed_by_id = {s["paper_id"]: s for s in seeds}
print(f"Retomando: {len(facts_by_id)} fatos, {len(drafts_by_id)} rascunhos já em cache.")

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def fit_extract_messages(seed):
    budget = MAX_MODEL_LEN - EXTRACT_MAX_TOKENS - PROMPT_SAFETY_MARGIN
    words = seed["introduction"].split()
    cut, truncated = len(words), False
    while True:
        msgs = build_extract_messages(seed, " ".join(words[:cut]))
        if backend.count_tokens(msgs) <= budget:
            return msgs, truncated
        if cut <= 300:
            return None, True
        cut, truncated = int(cut * 0.85), True

fail_reasons, truncated_inputs = Counter(), set()
t0 = time.time()
def out_of_time():
    return (time.time() - t0) > MAX_RUNTIME_HOURS * 3600

stopped_early = False
for rnd in range(1, MAX_ROUNDS + 1):
    # ---------- Etapa 1: fatos (Prompt 1) ----------
    todo = [s for s in seeds if s["paper_id"] not in facts_by_id]
    for batch in tqdm(list(chunks(todo, BATCH_SIZE)), desc=f"[r{rnd}] etapa 1 (fatos)", leave=False):
        if out_of_time():
            stopped_early = True
            break
        ready = []
        for s in batch:
            msgs, trunc = fit_extract_messages(s)
            if msgs is None:
                fail_reasons["prompt1_excede_contexto"] += 1
                continue
            if trunc:
                truncated_inputs.add(s["paper_id"])
            ready.append((s, msgs))
        if not ready:
            continue
        outs = backend.chat([m for _, m in ready], EXTRACT_TEMPERATURE, 0.95, EXTRACT_MAX_TOKENS, seed=SEED_RNG + rnd)
        new = []
        for (s, _), (text, cut_off) in zip(ready, outs):
            if cut_off:
                fail_reasons["etapa1_saida_truncada"] += 1
                continue
            f = clean_facts(parse_json_obj(text))
            if f is None:
                fail_reasons["facts_invalidos"] += 1
            else:
                facts_by_id[s["paper_id"]] = f
                new.append({"paper_id": s["paper_id"], "facts": f})
        append_jsonl(FACTS_CACHE, new)

    # ---------- Etapa 2: redação (Prompt 2) ----------
    todo = [s for s in seeds if s["paper_id"] in facts_by_id and s["paper_id"] not in drafts_by_id]
    for batch in tqdm(list(chunks(todo, BATCH_SIZE)), desc=f"[r{rnd}] etapa 2 (redação)", leave=False):
        if out_of_time():
            stopped_early = True
            break
        ready = []
        for s in batch:
            msgs = build_draft_messages(facts_by_id[s["paper_id"]])
            if backend.count_tokens(msgs) > MAX_MODEL_LEN - DRAFT_MAX_TOKENS - PROMPT_SAFETY_MARGIN:
                fail_reasons["prompt2_excede_contexto"] += 1
                facts_by_id.pop(s["paper_id"], None)   # força nova extração (que tende a ser mais concisa)
                continue
            ready.append((s, msgs))
        if not ready:
            continue
        outs = backend.chat([m for _, m in ready], DRAFT_TEMPERATURE, DRAFT_TOP_P, DRAFT_MAX_TOKENS, seed=SEED_RNG + 1000 * rnd)
        ts, new = now_iso(), []
        for (s, _), (text, cut_off) in zip(ready, outs):
            if cut_off:
                fail_reasons["etapa2_saida_truncada"] += 1
                continue
            secs, why = validate_draft(parse_json_obj(text), s)
            if secs is None:
                fail_reasons[why] += 1
                continue
            rec = {"paper_id": s["paper_id"], "sections": secs, "timestamp": ts, "round": rnd}
            drafts_by_id[s["paper_id"]] = rec
            new.append(rec)
        append_jsonl(DRAFTS_CACHE, new)

    pending = len(seeds) - len(drafts_by_id)
    print(f"Rodada {rnd}: {len(drafts_by_id)}/{len(seeds)} aprovados, {pending} pendentes "
          f"({(time.time()-t0)/60:.1f} min). Falhas acumuladas: {dict(fail_reasons)}")
    if pending == 0 or stopped_early:
        break
if stopped_early:
    print(f"TEMPO LIMITE ({MAX_RUNTIME_HOURS} h) atingido: progresso salvo em {CACHE_DIR}. Reexecute o notebook para continuar.")
print(f"Sementes cuja introdução foi encurtada para caber no contexto: {len(truncated_inputs)}")
print(f"Saídas truncadas por max_tokens (devem ser 0): "
      f"{fail_reasons['etapa1_saida_truncada'] + fail_reasons['etapa2_saida_truncada']}")

## 8. Montagem dos registros no esquema da seção 3 e gravação

In [ ]:
def build_record(seed, draft):
    s = draft["sections"]
    a, i, c = wc(s["abstract"]), wc(s["introduction"]), wc(s["conclusion"])
    return {
        "paper_id": f"synthetic::{seed['paper_id']}",
        "doi": None,
        "title": seed["title"],          # único campo herdado da semente
        "authors": None,
        "publication_year": None,
        "conference": None,
        "url": None,
        "abstract": s["abstract"],
        "introduction": s["introduction"],
        "conclusion": s["conclusion"],
        "section_lengths": {"abstract_words": a, "intro_words": i, "conclusion_words": c, "total_words": a + i + c},
        "provenance_type": OUTPUT_PROVENANCE_TYPE,
        "label": 1,
        "generator_metadata": {
            "model_name": backend.model_name,
            "temperature": DRAFT_TEMPERATURE,
            "top_p": DRAFT_TOP_P,
            "prompt_id": f"{PROMPT_ID_EXTRACT}+{PROMPT_ID_DRAFT}",
            "timestamp": draft["timestamp"],
        },
    }

records = [build_record(seed_by_id[pid], d) for pid, d in drafts_by_id.items() if pid in seed_by_id]
records.sort(key=lambda r: r["paper_id"])

# --- validação final do esquema ---
SCHEMA_KEYS = ["paper_id", "doi", "title", "authors", "publication_year", "conference", "url", "abstract",
               "introduction", "conclusion", "section_lengths", "provenance_type", "label", "generator_metadata"]
for r in records:
    assert list(r.keys()) == SCHEMA_KEYS
    assert r["label"] == 1 and r["provenance_type"] == OUTPUT_PROVENANCE_TYPE
    assert r["authors"] is None and r["publication_year"] is None and r["conference"] is None and r["url"] is None and r["doi"] is None
    gm = r["generator_metadata"]
    assert set(gm) == {"model_name", "temperature", "top_p", "prompt_id", "timestamp"}
    assert all(gm[k] is not None for k in gm)
    assert not any(ch in r[k] for k in ("abstract", "introduction", "conclusion") for ch in "\x08\x0c\x0b")  # LaTeX corrompido
assert len({r["paper_id"] for r in records}) == len(records)

missing_ids = [s["paper_id"] for s in seeds if s["paper_id"] not in drafts_by_id]
if missing_ids:
    partial = OUTPUT_PATH + ".partial"
    with open(partial, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print("=" * 78)
    print(f"INCOMPLETO: {len(records)}/{len(seeds)} registros. {len(missing_ids)} sementes sem texto aprovado "
          f"(ex.: {missing_ids[:3]}).")
    print(f"Parcial salvo em {partial} (NÃO é o arquivo final). Reexecute a célula 7 (ou o notebook): retoma do cache.")
    print("Motivos de falha acumulados:", dict(fail_reasons))
    print("=" * 78)
else:
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    if os.path.exists(OUTPUT_PATH + ".partial"):
        os.remove(OUTPUT_PATH + ".partial")
    print(f"OK: {len(records)} registros sintéticos gravados em {OUTPUT_PATH} (uma contraparte por semente).")

fact_records = [
    {
        "paper_id": pid,
        "title": seed_by_id[pid]["title"],
        "facts": facts_by_id[pid],
        "prompt_id": PROMPT_ID_EXTRACT,
    }
    for pid in sorted(facts_by_id)
    if pid in seed_by_id
]
with open(FACTS_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in fact_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"OK: {len(fact_records)} registros de fatos (Prompt 1) gravados em {FACTS_OUTPUT_PATH}.")


## 9. Resumo de sanidade

In [ ]:
if not records:
    print('Nenhum registro aprovado ainda  -  reexecute a célula 7 para continuar.')
else:
    def seed_of(r):
        return seed_by_id[r["generator_metadata"] and r["paper_id"].split("synthetic::", 1)[1]]

    df = pd.DataFrame([{
        "conference": seed_of(r)["conference"], "year": seed_of(r)["publication_year"],   # estrato vem da semente (não do registro)
        **r["section_lengths"]} for r in records])
    print("Registros por estrato da semente (conferência x ano):")
    print(df.groupby(["conference", "year"]).size().unstack(fill_value=0))
    print("\nPalavras por seção (sintético):")
    print(df[["abstract_words", "intro_words", "conclusion_words", "total_words"]].describe().round(1))

    hum = pd.DataFrame([{"abstract_words": wc(s["abstract"]), "intro_words": wc(s["introduction"]),
                         "conclusion_words": wc(s["conclusion"])} for s in seeds])
    print("\nMédia de palavras  -  humano (sementes) vs IA:")
    print(pd.DataFrame({"humano": hum.mean(), "ia": df[hum.columns].mean()}).round(1))

    print("\nAmostra:")
    r0 = records[0]
    print(r0["paper_id"], "|", r0["title"])
    print(r0["abstract"][:500], "...")